In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [ ]:
expts = {'XLI2':'Schisto',
         'XCE1':'CoV',
         'JYH3':'BoNT/A',
         'final':'Test'}
expt_order = ['XLI2','JYH3','XCE1','final']
alg_order = [('nanomap', 'individual'),
             ('nanomap', 'meta'),
             ('scoper', 'individual'),
             ('mmseqs', 'individual')]

scores = pd.read_csv('../fig3_meta-clustering_validation/all_scores.csv',index_col=0)
scores

In [ ]:
cut_vals = {
    'single':np.linspace(0.05,0.4,10),
    'average':np.linspace(0.05,0.8,10),
    'complete':np.linspace(0.05,0.95,10),
}
methods = ['single','average','complete']
merge_cuts = np.linspace(0.1,0.25,4)


def get_agg_method(name):
    s = name.split('_')
    if 'clone_id' in name:
        if s[0] == 'meta':
            return s[3]
        else:
            return methods[int(s[2])]
    else:
        return s[1]

def get_dist_cut(name):
    s = name.split('_')
    if 'clone_id' in name:
        if s[0] == 'meta':
            return float(s[-1])
        else:
            m = methods[int(s[2])]
            return round(cut_vals[m][int(s[3])],2)
    else:
        return float(s[-1])

def get_merge_cut(name):
    s = name.split('_')
    if 'clone_id' in name:
        if s[0] == 'meta':
            return None
        else:
            if len(s) == 4:
                return 0
            else:
                return round(merge_cuts[int(s[-1])],2)
    else:
        return None

scores['agg_method'] = scores['name'].map(get_agg_method)
scores['max_dist'] = scores['name'].map(get_dist_cut)
scores['merge_cut'] = scores['name'].map(get_merge_cut)

scores

In [ ]:
ylims = {
    'ari':[-0.05,1.05],
    'silhouette':[-0.02,0.48],
    'ranksum_score':[0,1.1],
}
letters='ABCDEFGHIJKLM'

styles = ['-','--',':']
method_order = {
    ('nanomap', 'individual'):['single','average','complete'],
    ('nanomap', 'meta'):['single','average','complete'],
    ('scoper', 'individual'):['single','average','complete'],
    ('mmseqs', 'individual'):['connected','setcover']
}

color_id = {
    ('nanomap', 'individual'):9,
    ('nanomap', 'meta'):0,
    ('scoper', 'individual'):1,
    ('mmseqs', 'individual'):2
}

alg_name = {
    ('nanomap', 'individual'):'NanoMAP (ind)',
    ('nanomap', 'meta'):'NanoMAP (meta)',
    ('scoper', 'individual'):'SCOPer',
    ('mmseqs', 'individual'):'MMseqs2'
}

name_convert = {
    'ari':'ARI',
    'silhouette':'Silhouette',
    'ranksum_score':'Normalized\nPhenotypic Quality',
    'stability':'Stability'
}

score_order = ['silhouette','ranksum_score','ari','stability']

colorblind = sns.color_palette("colorblind")
tab10 = plt.get_cmap('tab10')

plt.figure(figsize=[13,17])
for e,expt in enumerate(expt_order):
    for s,sc in enumerate(score_order[:-1]):
        ax = plt.subplot(4,3,3*e + s + 1)

        for a,alg in enumerate(alg_order):
            for m,method in enumerate(method_order[alg]):
                if (alg[1]=='individual') and (alg[0]=='nanomap'):
                    subset = scores[(scores['factor']==1)&(scores['expt']==expt)&(scores['method']==alg[0])&(scores['kind']==alg[1])\
                                    &(scores['agg_method']==method)&(scores['merge_cut']==0.25)]
                else:
                    subset = scores[(scores['factor']==1)&(scores['expt']==expt)&(scores['method']==alg[0])\
                                    &(scores['kind']==alg[1])&(scores['agg_method']==method)]

                subset = subset.copy().sort_values(by='name')
                #normalize renksum score
                if sc == 'ranksum_score':
                    norm = scores.loc[(scores['factor']==1)&(scores['expt']==expt)&(scores['method']=='nanomap')\
                                        &(scores['kind']=='individual'),'ranksum_score'].max()
                    subset['ranksum_score'] = subset['ranksum_score']/norm
                    
                plt.plot(subset['max_dist'],subset[sc],styles[m],color=colorblind[color_id[alg]],linewidth=3,label=f'{alg_name[alg]}, {method}')
    
        if s==0 and e==3:
            plt.ylabel('Score',loc='bottom',fontsize=16)
            ax.annotate("", xytext=(-0.22, 0.27), xy=(-0.22, 4.9), xycoords='axes fraction',
                arrowprops={'width':1.7,'color':'k'})
            
            plt.xlabel('Max. Dist.',loc='left',fontsize=16)
            ax.annotate("", xytext=(0.4, -0.175), xy=(3.5, -0.175), xycoords='axes fraction',
                arrowprops={'width':1.7,'color':'k'})
        else:
            plt.ylabel('')
            plt.xlabel('')
    
        if sc == 'silhouette':
            plt.yticks([0,0.1,0.2,0.3,0.4])

        plt.ylim(ylims[sc])
        plt.yticks(fontsize=16)

        plt.legend([],frameon=False)

        if s==2:
            if expt == 'final':
                color = 'gray'
            else:
                color = tab10(e+3)
            ax.text(1.05, 0.5 - 0.033*len(expts[expt]), expts[expt], fontsize=20, rotation=-90, 
                    transform=ax.transAxes, color=color, fontweight='bold')
    
        plt.xlim([0,1])
        plt.xticks(fontsize=16)
        
        ax.text(-0.15, 1.05, letters[3*e + s], fontsize=32, transform=ax.transAxes)

        if e==0:
            plt.title(name_convert[sc],fontsize=20,y=1.05)

plt.subplots_adjust(hspace=0.3,wspace=0.25)
plt.legend(loc=[1.3,1.9],fontsize=12)
plt.savefig('figS1.png',dpi=300,bbox_inches='tight')